# Notebook 1 — Spectral Operators & Initial Data

**Series: Optimal Mixing of Passive Scalars** | [`mixing.py`](mixing.py)

This notebook builds the spectral machinery step by step: wavenumber arrays,
derivative operators, inverse Laplacian, and the four families of initial data.
It makes every line of `build_operators` and `idata_*` transparent.

| Notebook | Content |
|---|---|
| **01 (this)** | **Spectral operators · initial data** |
| 02 | ODE right-hand side · simulation · norm analysis |


## 1  The Physical Problem

We mix a passive scalar $\theta(x,y,t)$ on the 2-torus $[0,1]^2$:

$$\partial_t \theta + (u \cdot \nabla)\theta = 0, \qquad \theta(\cdot,0)=\theta_0.$$

The velocity $u$ is the **Lin–Thiffeault–Doering optimal mixing velocity**:

$$v = -\Delta^{-1} P\!\left(\theta\,\nabla\Delta^{-1}\theta\right),
\qquad u = F\,\frac{v}{\|\nabla v\|_{L^2}},$$

where $P$ is the **Leray projection** onto divergence-free vector fields,
$F>0$ is the enstrophy constraint $\|\nabla u\|_{L^2}=F$, and the
**$H^{-1}$ mix norm** quantifies mixing:

$$\|\theta\|_{H^{-1}}^2 = \sum_{k\neq 0}\frac{|\hat\theta(k)|^2}{4\pi^2|k|^2}.$$

The mix norm decreases monotonically; when it is small, the scalar has been
spread uniformly over many scales (well mixed).


## 2  Grid Setup and FFT Convention

We discretise $[0,1]^2$ on an $N\times N$ grid with spacing $dx=1/N$:

$$x_j = j\,dx,\quad y_i = i\,dx, \qquad 0\le i,j\le N-1.$$

Using `numpy.meshgrid(x, x)` produces  
- `xx[i,j] = x[j]` — column index $j$ gives the $x$-coordinate  
- `yy[i,j] = x[i]` — row index $i$ gives the $y$-coordinate  

The 2-D DFT (`numpy.fft.fft2`) convention:

$$\hat f[k_y, k_x] = \sum_{i,j} f[i,j]\,e^{-2\pi i(ik_y+jk_x)/N}.$$

**Parseval identity:**
$\displaystyle\|\hat f\|_{\ell^2}^2 = N^2\,\|f\|_{\ell^2}^2,$ so
$\|f\|_{L^2} = \|\hat f\|_{\ell^2}/N^2$.


In [ ]:
import sys, os
sys.path.insert(0, '.')   # mixing.py lives in the same directory

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

N  = 32
dx = 1.0 / N
x  = np.arange(N) * dx
xx, yy = np.meshgrid(x, x)   # xx[i,j] = x[j],  yy[i,j] = x[i]

print(f'Grid : {N}×{N},  dx = {dx:.4f}')
print(f'xx[0, :5] = {xx[0,:5]}   ← x-values for first row')
print(f'yy[:5, 0] = {yy[:5,0]}   ← y-values for first column')


## 3  Wavenumber Construction

NumPy's `fft2` stores the output with wavenumber 0 first:

$$k = [0,\,1,\,2,\,\ldots,\,N/2,\;N/2+1,\,\ldots,\,N-1].$$

But wavenumbers above $N/2$ correspond to **negative** frequencies
(e.g. $k = N-1$ aliases to $k = -1$). We centre them:

$$k_{\text{eff}}[j] = \begin{cases}j & j < N/2 \\ j - N & j > N/2 \\ N/2 & j = N/2\end{cases}$$

This is exactly `k - N*(k > N//2)` in NumPy (matching `k - N*(k > N/2)` in MATLAB).

### Nyquist mode ($k = N/2$)

For even $N$ the mode at index $N/2$ is **ambiguous** — it is equally  
$+N/2$ and $-N/2$.  We keep it for the **Laplacian** (second derivative,  
even in $k$), but **zero it for first-derivative operators** to avoid aliasing.


In [ ]:
k     = np.arange(N)
k_eff = (k - N * (k > N // 2)).astype(float)   # centred wavenumbers

print('Index:  ', k)
print('k_eff:  ', k_eff.astype(int))
print()
print(f'Nyquist index: {N//2}   k_eff[{N//2}] = {k_eff[N//2]:.0f}')
print(f'Negative freqs start at index {N//2+1}: k_eff = {k_eff[N//2+1:].astype(int)}')


In [ ]:
# ── Verification: differentiate sin(2π·2·x) spectrally ──────────────────────
# Exact derivative of sin(2π·2·x) is 2π·2·cos(2π·2·x)

f_1d  = np.sin(2*np.pi*2*x)            # signal
f_hat = np.fft.fft(f_1d)               # DFT

k_d   = k_eff.copy()
k_d[N//2] = 0.0                        # zero Nyquist for first derivative
df_hat = (2j * np.pi * k_d) * f_hat   # multiply by 2πi·k_eff
df_num = np.real(np.fft.ifft(df_hat)) # back to physical space
df_exact = 2*np.pi*2 * np.cos(2*np.pi*2*x)

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(x, f_1d, label=r'$f = \sin(4\pi x)$')
axes[0].plot(x, df_num,   '--', label='spectral $df/dx$')
axes[0].plot(x, df_exact, ':', lw=2, label='exact $df/dx$')
axes[0].legend(); axes[0].set_xlabel('x')
axes[0].set_title('Spectral first derivative')

err = np.abs(df_num - df_exact)
axes[1].semilogy(x, err + 1e-17)
axes[1].set_xlabel('x'); axes[1].set_ylabel('|error|')
axes[1].set_title(f'Max error = {err.max():.2e}  (machine precision)')
plt.tight_layout(); plt.show()


## 4  The Derivative Operators `DEL_X` and `DEL_Y`

In 2D, multiplying the DFT by $2\pi i\,k_x$ (the *column* wavenumber)  
gives the $x$-derivative; multiplying by $2\pi i\,k_y$ (the *row* wavenumber)  
gives the $y$-derivative.

We store these as broadcastable arrays:

| Array | Shape | `DEL[i,j]` | Acts on |
|---|---|---|---|
| `DEL_X` | `(1, N)` | $2\pi i\,k_d[j]$ | columns ($x$-direction) |
| `DEL_Y` | `(N, 1)` | $2\pi i\,k_d[i]$ | rows ($y$-direction) |

When you write `DEL_X * theta_hat`, NumPy broadcasts `(1,N)` over `(N,N)`,  
which is exactly the column-wise multiplication that MATLAB achieves  
with a full $N\times N$ matrix `del_x` where every row is the same.


In [ ]:
k_d   = k_eff.copy()
k_d[N//2] = 0.0                          # Nyquist zeroed for first derivatives

DEL_X = (2j * np.pi * k_d)[np.newaxis, :]   # shape (1, N)
DEL_Y = (2j * np.pi * k_d)[:, np.newaxis]   # shape (N, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, arr, title in zip(axes,
        [DEL_X.ravel().imag, DEL_Y.ravel().imag],
        ['DEL_X (imaginary part, broadcast over rows)',
         'DEL_Y (imaginary part, broadcast over cols)']):
    ax.plot(arr, 'o-', ms=4)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel('wavenumber index')
    ax.set_ylabel(r'$2\pi k$')
    ax.set_title(title)
plt.tight_layout(); plt.show()
print(f'DEL_X shape: {DEL_X.shape}  DEL_Y shape: {DEL_Y.shape}')
print(f'Nyquist mode (index {N//2}) is zero: DEL_X[0,{N//2}] = {DEL_X[0,N//2]}')


## 5  Inverse Laplacian `LAP_INV`

In Fourier space the Laplacian acts as

$$\widehat{\Delta f}[k_y,k_x] = \bigl((2\pi i\,k_x)^2 + (2\pi i\,k_y)^2\bigr)\,\hat f
= -(2\pi)^2|k|^2\,\hat f.$$

`LAP_INV` is the element-wise inverse:

$$\text{LAP\_INV}[k_y,k_x] = \frac{-1}{(2\pi)^2|k|^2},\qquad
\text{LAP\_INV}[0,0] = 0.$$

The DC mode ($k=0$) is excluded — it represents the spatial mean of $\theta$,  
which is conserved and unconstrained by the transport.

`LAP_INV` is a **real, non-positive** $N\times N$ array; its most negative  
values are at low wavenumbers (long spatial scales).


In [ ]:
kx  = (2j * np.pi * k_eff)[np.newaxis, :]   # shape (1, N), uses k_eff (not k_d)
ky  = (2j * np.pi * k_eff)[:, np.newaxis]   # shape (N, 1)
lap = kx**2 + ky**2                          # -(2π)²|k|², shape (N, N)

LAP_INV = np.zeros_like(lap)
nz = lap != 0
LAP_INV[nz] = 1.0 / lap[nz]   # LAP_INV[0,0] stays 0 (DC mode excluded)

print(f'lap is real-valued: {np.allclose(lap.imag, 0)}')
print(f'LAP_INV is real-valued: {np.allclose(LAP_INV.imag, 0)}')
print(f'LAP_INV[0,0] = {LAP_INV[0,0]}  (DC mode excluded)')
print(f'LAP_INV[0,1] = {LAP_INV[0,1].real:.6f}  (k=(0,1): -1/(2π)² = {-1/(4*np.pi**2):.6f})')
print(f'min(LAP_INV) = {LAP_INV.real.min():.4f}  at low-k corner')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(np.fft.fftshift(LAP_INV.real), cmap='plasma',
               extent=[-N//2, N//2, -N//2, N//2], origin='lower')
plt.colorbar(im, ax=ax)
ax.set_xlabel('$k_x$'); ax.set_ylabel('$k_y$')
ax.set_title('LAP_INV (fftshift view)')
plt.tight_layout(); plt.show()


## 6  The Mix-Norm Operator `LAMBDA_INV`

The $H^{-1}$ mix norm is

$$\|\theta\|_{H^{-1}}^2 = \sum_{k\neq 0}\frac{|\hat\theta(k)|^2}{4\pi^2|k|^2}
= \|\lambda^{-1}\,\hat\theta\|_{\ell^2}^2,\qquad\lambda = 2\pi|k|.$$

So $\lambda^{-1}[k] = \frac{1}{2\pi|k|} = \sqrt{-\text{LAP\_INV}[k]}$.

In code: `LAMBDA_INV = np.sqrt(np.where(LAP_INV != 0, -LAP_INV, 0))`.

It is a **real, non-negative** $N\times N$ array — largest at low wavenumbers  
(long spatial scales are weighted most in the mix norm).


In [ ]:
LAMBDA_INV = np.sqrt(np.where(LAP_INV != 0, -LAP_INV, 0.0))

print(f'LAMBDA_INV is real non-negative: min={LAMBDA_INV.min():.4f}')
print(f'LAMBDA_INV[0,1] = {LAMBDA_INV[0,1].real:.6f}  (= 1/(2π)·1 = {1/(2*np.pi):.6f})')

# Verify: ||lambda_inv * theta_hat||_2  is the H^{-1} norm (up to normalisation)
f = np.sin(2*np.pi*xx) * np.sin(2*np.pi*yy)
f_hat = np.fft.fft2(f)
hm1_sq = np.sum(np.abs(LAMBDA_INV * f_hat)**2) / N**4   # Parseval factor
hm1_analytic = 1 / (2 * (2*np.pi)**2 * 2)               # exact for sin(2πx)sin(2πy)
print(f'\nH^(-1) norm² of sin(2πx)sin(2πy): numerical={hm1_sq:.6f},  analytic={hm1_analytic:.6f}')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(np.fft.fftshift(LAMBDA_INV.real), cmap='viridis',
               extent=[-N//2, N//2, -N//2, N//2], origin='lower')
plt.colorbar(im, ax=ax)
ax.set_xlabel('$k_x$'); ax.set_ylabel('$k_y$')
ax.set_title(r'$\lambda^{-1}$ (LAMBDA_INV, fftshift view)')
plt.tight_layout(); plt.show()


## 7  `build_operators` — Putting It All Together

`build_operators(N)` wraps all the above into a single dictionary:

```python
ops = {
    'N'          : N,           # grid size
    'dx'         : 1/N,         # grid spacing
    'DEL_X'      : ...,         # shape (1,N)  — x-derivative multiplier
    'DEL_Y'      : ...,         # shape (N,1)  — y-derivative multiplier
    'LAP_INV'    : ...,         # shape (N,N)  — Δ⁻¹ multiplier (real)
    'LAMBDA_INV' : ...,         # shape (N,N)  — λ⁻¹ for H⁻¹ norm (real)
    'xx'         : ...,         # shape (N,N)  — x-coordinates mesh
    'yy'         : ...,         # shape (N,N)  — y-coordinates mesh
}
```

The key design point: **DEL_X and DEL_Y use `k_d` (Nyquist-zeroed) while  
LAP_INV uses `k_eff` (full wavenumbers).**  This is because:
- First derivatives are ambiguous at the Nyquist frequency → zero it.  
- The Laplacian is $k^2$ (even function) → the Nyquist mode is well-defined.


In [ ]:
from mixing import build_operators

N   = 32
ops = build_operators(N)

print('Keys in ops:', list(ops.keys()))
for key, val in ops.items():
    if isinstance(val, np.ndarray):
        print(f'  {key:12s}: shape={str(val.shape):10s}  dtype={val.dtype}')
    else:
        print(f'  {key:12s}: {val}')

# Confirm DEL_X uses Nyquist-zeroed wavenumber
print(f'\nNyquist mode check: DEL_X[0,{N//2}] = {ops["DEL_X"][0,N//2]}  (should be 0)')
print(f'LAP_INV at Nyquist corner [N//2,N//2] = {ops["LAP_INV"][N//2,N//2].real:.6f}')


## 8  Initial Data Functions

All four initial data functions take arguments `(a, ops)` and return a  
**real, $L^2$-normalised** $N\times N$ array supported (approximately) near  
the centre of $[0,1]^2$, where $a\in(0,1)$ controls the support size.

| Function | Support | Shape |
|---|---|---|
| `idata_sin` | $[0,a]^2$, centred | Unimodal |
| `idata_diag` | Two sub-squares of $[0,a]^2$ | Asymmetric pair |
| `idata_strip` | $[0,a]\times[0,a/2]$, centred | Aspect-ratio $2:1$ |
| `idata_trigpoly` | All of $[0,1]^2$ | Multifrequency |

**$L^2$ normalisation:** each function divides by $\|f\|_{\ell^2}\cdot dx$,  
so that the continuous $L^2$ norm is 1:  
$\|\theta_0\|_{L^2} = \|\theta_0\|_{\ell^2}\cdot dx \approx 1$.

The support is **circularly shifted** (`np.roll`) to the centre after  
construction, so that mixing is not biased by proximity to a boundary.


In [ ]:
from mixing import idata_sin, idata_diag, idata_strip, idata_trigpoly

N   = 64
ops = build_operators(N)

# Effect of the scale parameter a: smaller a → narrower support
a_vals = [0.25, 0.50, 0.75]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, a in zip(axes, a_vals):
    theta0 = idata_sin(a, ops)
    im = ax.imshow(theta0, origin='lower', extent=[0,1,0,1], cmap='RdBu_r', aspect='equal',
                   vmin=-theta0.max(), vmax=theta0.max())
    ax.set_title(f'idata_sin,  a={a}', fontsize=10)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(r'$\theta_0$ for different scale parameters $a$', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# All four initial data functions at a = 0.5
a = 0.5
fns = [
    (idata_sin,      r'$\sin(2\pi x/a)\sin(2\pi y/a)\cdot\mathbf{1}_{[0,a]^2}$'),
    (idata_diag,     'Diagonal (two offset sub-squares)'),
    (idata_strip,    r'Strip  $[0,a]\times[0,a/2]$'),
    (idata_trigpoly, r'Trig polynomial  $\sum c_k \sin(2k\pi x/a)\sin(2k\pi y/a)$'),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, (fn, label) in zip(axes, fns):
    theta0 = fn(a, ops)
    vmax = np.abs(theta0).max()
    im = ax.imshow(theta0, origin='lower', extent=[0,1,0,1],
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='equal')
    ax.set_title(label, fontsize=8, wrap=True)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f'Four initial data functions  (a={a})', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Verify L2 normalisation for all functions and several a values
print('L2 norms (should all be ≈ 1.00):\n')
print(f'{"function":<16}  {"a=0.25":>8}  {"a=0.50":>8}  {"a=0.75":>8}')
print('-'*46)
for fn in [idata_sin, idata_diag, idata_strip, idata_trigpoly]:
    row = [f'{np.linalg.norm(fn(a, ops).ravel())*ops["dx"]:.6f}' for a in [0.25,0.50,0.75]]
    print(f'{fn.__name__:<16}  {"  ".join(row)}')


## Summary — Notebook 1

| Concept | Key formula | Code |
|---|---|---|
| Wavenumber | $k_{\text{eff}} = k - N\cdot(k > N/2)$ | `k - N*(k > N//2)` |
| First derivative | $\times\,2\pi i\,k_{\text{eff}}$, Nyquist $\to 0$ | `DEL_X`, `DEL_Y` |
| Inverse Laplacian | $\times\,(-4\pi^2|k|^2)^{-1}$, DC $\to 0$ | `LAP_INV` |
| Mix-norm weight | $\lambda^{-1} = 1/(2\pi|k|) = \sqrt{-\text{LAP\_INV}}$ | `LAMBDA_INV` |
| $L^2$ norm via Parseval | $\|f\|_{L^2} = \|\hat f\|_{\ell^2}/N^2$ | `norm(f_hat)/N**2` |
| Initial data | 4 families, $L^2$-normalised, centred | `idata_*` |

**Notebook 2** explains the ODE right-hand side (how $u$ is computed),
the resolution check event, and the full simulation and analysis pipeline.
